In [1]:
import caf.base as cb
import caf.tem as ct
import pandas as pd
import os
from pathlib import Path

# HB Production
## Preprocessing
### Create equivalent population DVector

In [3]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\lu_pop_2023.hdf"
if not os.path.exists(dvec_path):
    pop = pd.read_csv(r"I:\NorMITs NoTEM\voa_gb_2023_uni\landuse\lu_pop_2023.csv")
    pop = pop.set_index(["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"])
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"],
                                            naming_order=["gender_3", "aws", "adult_nssec", "ns_sec", "soc", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("lsoa_2021")
    pop_dvec = cb.DVector(segmentation=segmentation, import_data=pop, zoning_system=zoning_system)
    pop_dvec.save(dvec_path)

### Create equivalent trip rates DVector

In [4]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\hb_trip_rates_production.hdf"
if not os.path.exists(dvec_path):
    tr = pd.read_csv(r"I:\NTS\outputs\productions\hb\trip_rates\hb_trip_rates_production.csv")
    tr = tr.rename(columns={"gender": "gender_3", "ns": "ns_sec", "purpose": "p"}).pivot(index=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"], columns="tfn_at", values="beta")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"],
                                                        naming_order=["gender_3", "aws", "ns_sec", "soc", "hh_type", "p"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    tr_dvec = cb.DVector(segmentation=segmentation, import_data=tr, zoning_system=zoning_system)
    tr_dvec.save(dvec_path)

### Create equivalent 2023 adjustment DVector

In [5]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\trip_rate_adjustments_production_hb_fr.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"I:\NTS\outputs\others\trip_rate_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p"],
                                                        naming_order=["p"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

### Create equivalent mts DVector

In [6]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\mode_time_split_production_hb_fr_reg.hdf"
if not os.path.exists(dvec_path):
    mts = pd.read_csv(r"I:\NTS\outputs\productions\hb\mode_time_splits\mode_time_split_production_hb_fr_reg.csv")
    mts = mts.rename(columns={"mode": "m", "period": "tp", "purpose": "p"}).pivot(index=["p", "m", "tp", "hh_type"], columns="tfn_at", values="rho")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "m", "tp", "hh_type"],
                                                        naming_order=["p", "m", "tp", "hh_type"]))
    zoning_system = cb.ZoningSystem.get_zoning("tfn_at")
    mts_dvec = cb.DVector(segmentation=segmentation, import_data=mts, zoning_system=zoning_system)
    mts_dvec.save(dvec_path)

### Create equivalent mts Adjustment DVector

In [7]:
dvec_path = r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\mode_time_split_adjustments.hdf"
if not os.path.exists(dvec_path):
    adj = pd.read_csv(r"I:\NTS\outputs\others\mode_time_split_adjustments.csv")
    adj = adj.rename(columns={"purpose": "p", "period": "tp", "mode": "m"}).loc[(adj["pa"]=="p") & (adj["direction"]=="hb_fr")].pivot(index=["p", "tp", "m"], columns="gor", values="adj")
    segmentation = cb.Segmentation(cb.SegmentationInput(enum_segments=["p", "tp", "m"],
                                                        naming_order=["p", "tp", "m"]))
    zoning_system = cb.ZoningSystem.get_zoning("gor")
    adj_dvec = cb.DVector(segmentation=segmentation, import_data=adj, zoning_system=zoning_system)
    adj_dvec.save(dvec_path)

## TEM Setup

In [8]:
tem = ct.TEM(
    model_years=[2023],
    scenario="Core",
    output_zoning="normits",
    iteration_name="HBProd_specific_test",
    export_home=r"T:\ThomasPrince\TEM I-Drive Comparison\Outputs - caf.tem",
    return_segmentation=["hh_type", "p", "m", "tp"]
)

## HB Production Model Setup and Run

In [9]:
input_dir = Path(r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction")

HBProd = tem.HBProductionModel(
    population_paths={2023: input_dir / "lu_pop_2023.hdf"},
    trip_rates_path=input_dir / "hb_trip_rates_production.hdf",
    mode_time_splits_path=input_dir / "mode_time_split_production_hb_fr_reg.hdf",
    adjustment_path=input_dir / "trip_rate_adjustments_production_hb_fr.hdf",
    mts_adj_path=r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\01-HBProduction\mode_time_split_adjustments.hdf",
    population_translation_path=r"T:\ThomasPrince\TEM I-Drive Comparison\Inputs\normits_lsoa_2021_pop.csv"
)

In [ ]:
if not ((os.path.exists(HBProd.model.export_paths.mts_demand_adj[2023])) and (os.path.exists(HBProd.model.export_paths.pure_demand[2023]))):
    HBProd.run(True, True, True, True) # Output matches sufficiently

In [ ]:
pure_demand = cb.DVector.load(HBProd.model.export_paths.pure_demand[2023])
pure_demand.aggregate(["p"]).data

In [ ]:
pop_2023 = pd.read_csv(r"I:\NorMITs NoTEM\voa_gb_2023_uni\reports\pop_2023_normits.csv")
pop_2023 = pop_2023.groupby(["normits_v3.3_id"])[["1","2","3","4","5","6","7","8"]].sum().T
pop_2023.index = pop_2023.index.astype(int)
pop_2023

In [ ]:
test = (pure_demand.aggregate(["p"]).data - pop_2023).stack().reset_index()
test.loc[abs(test[0])>1]

There's one normits zone to look into...\
5248009\
NB. this normits zone has two tfn_at's - this is where the error will stem from...

NB. Pure Demand from HB Production matches the mdlnotem report "pop_2023_normits" perfectly!

In [ ]:
mts_demand_adj_prod = cb.DVector.load(HBProd.model.export_paths.mts_demand_adj[2023])
mts_demand_adj_prod.data

In [14]:
dvec_path = Path(r"T:\ThomasPrince\TEM I-Drive Comparison\Outputs - mdlnotem\hb_fr_prod.hdf")
if not os.path.exists(dvec_path):
    hb_fr = pd.read_csv(r"I:\NorMITs NoTEM\voa_gb_2023_uni\tripend\NorMITs_tripend_2023_hb_fr.csv.bz2")
    hb_fr_prod = hb_fr.rename(columns={"purpose": "p", "mode": "m", "period": "tp", "ns": "ns_sec"}).groupby(["normits_v3.3_id", "p", "m", "tp", "ns_sec", "soc"])[["prod"]].sum().reset_index().pivot(index = ["p", "m", "tp", "ns_sec", "soc"], columns="normits_v3.3_id", values="prod")
    segmentation = cb.Segmentation(cb.segmentation.SegmentationInput(enum_segments=["p", "m", "tp", "ns_sec", "soc"], naming_order=["p", "m", "tp", "ns_sec", "soc"]))
    hb_fr_prod_dvec = cb.DVector(segmentation=segmentation, import_data=hb_fr_prod, zoning_system=cb.ZoningSystem.get_zoning("normits"))
    hb_fr_prod_dvec.save(dvec_path)
hb_fr_prod = cb.DVector.load(dvec_path)

In [ ]:
hb_fr_prod.data.head(5)

In [ ]:
mts_demand_adj_prod.aggregate(hb_fr_prod.segmentation).data.head(5)

In [17]:
comparison_abs = mts_demand_adj_prod.aggregate(hb_fr_prod.segmentation) - hb_fr_prod
hb_fr_prod.fill(0, 1)
comparison_prop = mts_demand_adj_prod.aggregate(hb_fr_prod.segmentation) / hb_fr_prod

In [18]:
comparison_abs.save(r"T:\ThomasPrince\TEM I-Drive Comparison\Comparison\absolute_difference_mts_adjusted.hdf")
comparison_prop.save(r"T:\ThomasPrince\TEM I-Drive Comparison\Comparison\proportional_difference_mts_adjusted.hdf")

All differences are within 0.01% except for the one expected differing zone and where mts_demand_adj trips are zero.

In [ ]:
test = comparison_prop.data.stack()
test.loc[((test<0.9999) & (test!=0)) | (test>1.0001)].reset_index()["normits_id"].unique()

In [ ]:
test = comparison_abs.data.stack()
test.loc[abs(test)>1].reset_index()["normits_id"].unique()